In [1]:
%run shared_imports.py
from decimal import Decimal
from dataclasses import dataclass
from sqlalchemy.orm.query import Query
import gspread
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from statistics import mode

In [11]:
%run ss13_walker.py

In [12]:
@dataclass(frozen=True)
class SpawnerData:
    feedback_key: str
    feedbacks: list[Feedback]

    def total_spawns(self):
        return sum((spawns for feedback in self.feedbacks for spawns in feedback.values()))

    def spawns_per_round(self):
        return [sum(x.values()) for x in self.feedbacks]

    def counts(self):
        counts = defaultdict(list)
        for feedback in self.feedbacks:
            for name, count in feedback.items():
                counts[name].append(count)
        return counts

    def average(self, path):
        return mean(x for x in self.counts().get(path, [0]))

    def spawn_paths(self):
        return {key for feedback in self.feedbacks for key in feedback.keys() if key}

    def min_round_count(self, path):
        return min(x for x in self.counts().get(path, [0]))

    def max_round_count(self, path):
        return max(x for x in self.counts().get(path, [0]))

    def mode(self, path):
        return mode(x for x in self.counts().get(path, [0]))

    def means(self):
        means = {name: round(Decimal(sum(count) / len(self.feedbacks)), 2) for name, count in self.counts().items()}
        return sorted(means.items(), key = lambda x: x[1])

    def total_count(self, path):
        return sum(x for x in self.counts().get(path, []))

    def effective_percent(self, path):
        return sum([x / self.total_spawns() for x in self.counts().get(path, [])])

In [3]:
engine = make_engine("settings.toml")

In [4]:
query = select(Feedback.json).join(Round).filter(
    and_(
        Feedback.round_id >= 41517,
        Feedback.key_name == "random_spawners",
        # Round.map_name == "Cyberiad",
    )
)

spawner_data = None
with Session(engine) as session:
    feedbacks = session.scalars(query)
    spawner_data = SpawnerData("random_spawners", [feedback['data'] for feedback in feedbacks.all()])
    # feedbacks = session.query(Feedback).join(Round).filter(Feedback.key_name==feedback_key)
    # return SpawnerData(feedback_key, feedbacks.all())


In [45]:
non_trash_totals = list()
for feedback in spawner_data.feedbacks:
    non_trash_totals.append(sum([y for x, y in feedback.items() if x != '/obj/item/trash']))

In [48]:
mean(non_trash_totals)

367.53125

405

In [14]:
ss13 = SS13("D:/ExternalRepos/third_party/Paradise/paradise.dme")


In [13]:
pre_ss13 = SS13("D:/ExternalRepos/third_party/ParadiseMaster/paradise.dme")

In [15]:
td = pre_ss13.dme.typedecl('/obj/effect/spawner/lootdrop/maintenance')

In [16]:
boxstation = DMM.from_file("D:/ExternalRepos/third_party/ParadiseMaster/_maps/map_files/stations/boxstation.dmm")

In [17]:
old_spawners = defaultdict(int)
for coord in boxstation.coords():
    tile = boxstation.tiledef(*coord)
    for spawner in tile.find("/obj/effect/spawner/lootdrop/maintenance"):
        old_spawners[tile.prefab_path(spawner)] += 1

In [18]:
old_spawners

defaultdict(int,
            {/obj/effect/spawner/lootdrop/maintenance/two: 65,
             /obj/effect/spawner/lootdrop/maintenance: 209,
             /obj/effect/spawner/lootdrop/maintenance/three: 19,
             /obj/effect/spawner/lootdrop/maintenance/eight: 5})

In [19]:
(5 * 8) + (19 * 3) + (65 * 2) + 209

436

In [30]:
runs_old = list()
lootcounts = dict()
for spawner in old_spawners.keys():
    lootcounts[spawner] = pre_ss13.dme.typedecl(spawner).value('lootcount')
loot_table = pre_ss13.dme.typedecl('/obj/effect/spawner/lootdrop/maintenance').value('loot')
for i in range(10):
    run = defaultdict(int)
    for spawner, count in old_spawners.items():
        for j in range(count):
            for k in range(lootcounts[spawner]):
                typepath = pick_weight_recursive(loot_table)
                if not typepath:
                    typepath = "/nothing"
                run[typepath] += 1
    runs_old.append(run)

In [31]:
pre_spawns = SpawnerData('random_spawners_pre', runs_old)

In [33]:
pre_spawns.total_count('/obj/item/extinguisher')

231

In [39]:
print("{:.2%}".format(pre_spawns.effective_percent('/obj/item/extinguisher')))

5.30%


In [44]:
results = defaultdict(int)
for i in range(int(mean(pre_spawn.spawns_per_round()))):
    results[pickweight(v)] += 1
    

In [5]:
pre_spawn = get_spawner_data('random_spawners_pre')
post_spawn = get_spawner_data('random_spawners_post')

In [ ]:
fake_feedback

In [6]:
mean(pre_spawn.spawns_per_round())

420.7

In [101]:
mean(post_spawn.spawns_per_round())

425.9

In [9]:
pre_spawn.effective_percent('/obj/item/extinguisher')

0.055383884002852386

In [10]:
post_spawn.effective_percent('/obj/item/extinguisher')

0.006701414743112435

In [150]:
pre_spawn.spawn_paths() - post_spawn.spawn_paths()

{'/obj/effect/spawner/random_spawners/mod/maint',
 '/obj/item/assembly/prox_sensor',
 '/obj/item/bio_chip_implanter/storage',
 '/obj/item/book/manual/wiki/engineering_construction',
 '/obj/item/book/manual/wiki/hacking',
 '/obj/item/clothing/gloves/color/yellow',
 '/obj/item/clothing/head/hardhat',
 '/obj/item/clothing/mask/chameleon',
 '/obj/item/clothing/suit/jacket/bomber/syndicate',
 '/obj/item/deck/cards/syndicate',
 '/obj/item/dnascrambler',
 '/obj/item/grenade/clown_grenade',
 '/obj/item/grenade/smokebomb',
 '/obj/item/gun/projectile/automatic/pistol',
 '/obj/item/hand_labeler',
 '/obj/item/melee/knuckleduster',
 '/obj/item/melee/knuckleduster/syndie',
 '/obj/item/mod/construction/broken_core',
 '/obj/item/paper/crumpled',
 '/obj/item/reagent_containers/spray/sticky_tar',
 '/obj/item/seeds/ambrosia/cruciatus',
 '/obj/item/soap/syndie',
 '/obj/item/stack/sheet/mineral/plasma',
 '/obj/item/stack/sheet/rglass',
 '/obj/item/stack/tape_roll',
 '/obj/item/stamp/chameleon',
 '/obj/item

In [78]:
sum([x / pre_spawn.total_spawns() for x in pre_spawn.counts()['/obj/item/dnascrambler']]) * 100


0.023769907297361538

In [138]:
pre_spawn.total_count('/obj/item/gun/projectile/automatic/pistol')

2

In [155]:
pre_spawn.spawn_paths() or post_spawn.spawn_paths()

{'/obj/effect/spawner/random_spawners/mod/maint',
 '/obj/item/ammo_box/magazine/m10mm',
 '/obj/item/assembly/prox_sensor',
 '/obj/item/assembly/timer',
 '/obj/item/bio_chip_implanter/storage',
 '/obj/item/bodybag',
 '/obj/item/book/manual/wiki/engineering_construction',
 '/obj/item/book/manual/wiki/hacking',
 '/obj/item/caution',
 '/obj/item/clothing/glasses/meson',
 '/obj/item/clothing/glasses/sunglasses',
 '/obj/item/clothing/gloves/color/black',
 '/obj/item/clothing/gloves/color/fyellow',
 '/obj/item/clothing/gloves/color/yellow',
 '/obj/item/clothing/gloves/color/yellow/fake',
 '/obj/item/clothing/head/cone',
 '/obj/item/clothing/head/hardhat',
 '/obj/item/clothing/head/hardhat/red',
 '/obj/item/clothing/head/that',
 '/obj/item/clothing/head/ushanka',
 '/obj/item/clothing/head/welding',
 '/obj/item/clothing/mask/chameleon',
 '/obj/item/clothing/mask/chameleon/voice_change',
 '/obj/item/clothing/mask/gas',
 '/obj/item/clothing/mask/gas/voice_modulator',
 '/obj/item/clothing/mask/gas

In [9]:
len(shared_paths)

105

In [6]:
gc = gspread.service_account()
sh = gc.open("Maint Loot Relootening")

In [7]:
worksheet = sh.get_worksheet(5)

In [73]:
worksheet = sh.add_worksheet(title="dataframes_output", rows=len(all_paths), cols=6)

In [10]:
set_with_dataframe(worksheet, df)

In [8]:
rows = []
for path in sorted(spawner_data.spawn_paths()):
    rows.append([
        path,
        spawner_data.min_round_count(path),
        spawner_data.mode(path),
        spawner_data.max_round_count(path),
        "{:.2%}".format(spawner_data.effective_percent(path)),
        # round(Decimal(spawner_data.effective_percent(path) * 100.0), 2),
        
    ])
df = pd.DataFrame(rows, columns=[
    "Type Path",
    "Minimum count",
    "Most common count",
    "Maximum count",
    f"Effective Percent ({len(spawner_data.feedbacks)} rounds)",
])

In [9]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(df)

,Type Path,Minimum count,Most common count,Maximum count,Effective Percent (64 rounds)
0,/obj/item/ammo_box/magazine/m10mm,1,1,3,0.15%
1,/obj/item/analyzer,1,6,19,1.19%
2,/obj/item/assembly/prox_sensor,1,1,6,0.44%
3,/obj/item/assembly/signaler,1,2,6,0.45%
4,/obj/item/assembly/timer,1,1,10,0.51%
5,/obj/item/assembly/voice,1,2,7,0.47%
6,/obj/item/assembly/voice/noise,1,2,7,0.47%
7,/obj/item/bio_chip_implanter/storage,1,1,1,0.02%
8,/obj/item/bodybag,1,2,8,0.49%
9,/obj/item/book,1,1,7,0.35%


In [120]:
len(post_spawn.feedbacks)

6

In [49]:
feedbacks = session.query(Feedback).join(Round).filter(Feedback.key_name=='random_spawners', Round.id >= 1938)

In [50]:
counts = defaultdict(list)
for feedback in feedbacks.all():
    for name, count in feedback.items():
        counts[name].append(count)

In [58]:
means = {name: round(decimal.Decimal(sum(count) / feedbacks.count()), 2) for name, count in counts.items()}

In [60]:
feedbacks.count()

74

In [61]:
for name, amt in :
    print(f"{name}: {amt}")

/obj/item/deck/cards/syndicate: 0.18
/obj/item/grenade/smokebomb: 0.26
/obj/item/gun/syringe/syndicate: 0.26
/obj/item/stamp/chameleon: 0.28
/obj/item/storage/secure/briefcase/syndie: 0.28
/obj/item/storage/backpack/satchel_flat: 0.30
/obj/item/clothing/suit/jacket/bomber/syndicate: 0.30
/obj/item/clothing/mask/chameleon: 0.32
/obj/item/grenade/clown_grenade: 0.32
/obj/item/storage/belt/military/traitor: 0.32
/obj/item/weaponcrafting/receiver: 0.32
/obj/item/storage/pill_bottle/fakedeath: 0.34
/obj/item/clothing/suit/storage/iaa/blackjacket/armored: 0.34
/obj/item/clothing/under/chameleon: 0.35
/obj/item/clothing/mask/gas/voice_modulator: 0.35
/obj/item/storage/backpack/duffel/syndie/med/surgery_fake: 0.38
/obj/item/suppressor: 0.38
/obj/item/ammo_box/magazine/m10mm: 0.39
/obj/item/clothing/mask/gas/voice_modulator/chameleon: 0.39
/obj/item/dnascrambler: 0.42
/obj/item/mod/construction/broken_core: 0.42
/obj/item/storage/toolbox/syndicate: 0.45
/obj/item/clothing/mask/chameleon/voice_c

In [71]:
engine = make_engine("settings.toml")
session = Session(engine)


In [45]:
session.query(Feedback).join(Round).where(
    and_(
        Feedback.key_name == 'testmerged_prs',
        Feedback.json["data"].regexp_match("25619"),
        Round.start_datetime >= datetime(2024,8,6)
    )
).count()


73

In [87]:
sum(session.query(Feedback).join(Round).filter(Feedback.key_name=='random_spawners', Round.id == 2027)[0].values())


555

In [76]:
sum(session.query(Feedback).join(Round).filter(Feedback.key_name=='random_spawners', Round.id == 2019)[0].values())


312

In [90]:
sum(session.query(Feedback).join(Round).filter(Feedback.key_name=='random_spawners', Round.id == 2029)[0].values())


142

In [106]:
with Session(engine) as session:
    feedbacks = session.query(Feedback).join(Round).filter(Feedback.key_name=='random_spawners', Round.id >= 2033)
    total_spawns = sum((spawns for feedback in feedbacks.all() for spawns in feedback.values()))
    counts = defaultdict(list)
    for feedback in feedbacks.all():
        for name, count in feedback.items():
            counts[name].append(count)
    means = {name: round(decimal.Decimal(sum(count) / feedbacks.count()), 2) for name, count in counts.items()}
    for name, amt in sorted(means.items(), key = lambda x: x[1]):
        print(f"{name}: {amt}")

/obj/item/storage/fancy/cigarettes/cigpack_syndicate: 0.50
/obj/item/grenade/smokebomb: 0.50
/obj/item/reagent_containers/dropper: 0.50
/obj/item/storage/backpack/satchel_flat: 0.50
/obj/item/seeds/ambrosia/cruciatus: 0.50
/obj/item/radio/off: 0.50
/obj/item/deck/cards/syndicate: 0.50
/obj/item/clothing/gloves/color/fyellow: 0.50
/obj/item/tank/internals/emergency_oxygen: 0.50
/obj/item/assembly/signaler: 0.50
/obj/item/reagent_containers/glass/bucket: 0.50
/obj/item/clothing/suit/storage/iaa/blackjacket/armored: 0.50
/obj/item/stamp/chameleon: 0.50
/obj/item/stock_parts/capacitor: 0.50
/obj/item/storage/pill_bottle/fakedeath: 0.50
/obj/item/storage/wallet: 0.50
/obj/item/stock_parts/micro_laser: 0.50
/obj/item/gun/projectile/automatic/pistol: 0.50
/obj/item/assembly/prox_sensor: 0.50
/obj/item/tank/internals/emergency_oxygen/engi: 0.50
/obj/item/storage/firstaid/regular: 0.50
/obj/item/reagent_scanner/adv: 0.50
/obj/item/melee/knuckleduster: 0.50
/obj/item/robotanalyzer: 0.50
/obj/ite